In [1]:
import networkx as nx
import json 
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# these are the two functions one needs to create a JSON file to upload and create the project in the backend 
import nx2json as nx2j
import uploaderGraph as uG

In [2]:
df_nodes_stacked = pd.read_csv("temp-files/students_celine/stacked/all_layer_nodes_stacked.csv")
df_edges_stacked = pd.read_csv("temp-files/students_celine/stacked/all_layers_edges_stacked.csv")
df_edges_stacked_inter = pd.read_csv("temp-files/students_celine/stacked/all_layers_edges_stacked_with_interlayer.csv")

In [3]:
df_nodes_stacked

,Node ID,x,y,z,r,g,b,a
0,ETV3__L01,9.184851e-17,1.500000,0.0,255,60,60,165
1,E2F3__L01,4.394907e-01,2.461066,0.0,255,60,60,173
2,NR3C1__L01,-4.394907e-01,2.461066,0.0,255,60,60,163
3,ZBTB14__L01,8.170588e-01,3.403295,0.0,255,60,60,168
4,ETV4__L01,2.143132e-16,3.500000,0.0,255,60,60,180
...,...,...,...,...,...,...,...,...
4739,PRPF31__L08,-1.073394e+01,9.000584,17.5,80,120,255,120
4740,TRAPPC4__L08,-1.099540e+01,8.764622,17.5,80,120,255,122
4741,MRPL1__L08,-1.060670e+01,8.775519,17.5,80,120,255,120
4742,PPID__L08,-1.087249e+01,8.407690,17.5,80,120,255,120


In [4]:
# make edgelists
l_edges_stacked = []
for index, row in df_edges_stacked.iterrows():
    l_edges_stacked.append((row['source'], row['target'], "stacked"))

l_edges_inter = []
for index, row in df_edges_stacked_inter.iterrows():
    l_edges_inter.append((row['source'], row['target'], "interlayer"))

In [5]:
# STACKED 
G_stacked = nx.Graph()
G_stacked.add_edges_from([(u, v, {"type": t}) for u, v, t in l_edges_stacked])
G_stacked.add_nodes_from(df_nodes_stacked['Node ID'].tolist()) 

# positions
pos_stacked = dict(zip(G_stacked.nodes(),list(zip(df_nodes_stacked['x'], df_nodes_stacked['y'], df_nodes_stacked['z']))))
nx.set_node_attributes(G_stacked, pos_stacked, 'pos')

# colors
col_stacked = dict(zip(G_stacked.nodes(),list(zip(df_nodes_stacked['r'], df_nodes_stacked['g'], df_nodes_stacked['b'], df_nodes_stacked['a']))))
nx.set_node_attributes(G_stacked, col_stacked, 'nodecolor')

print("number of nodes in actin graph: ", len(G_stacked.nodes()))
print("number of edges in actin graph: ", len(G_stacked.edges()))

number of nodes in actin graph:  4744
number of edges in actin graph:  6628


In [6]:
col_stacked

{'AHR__L01': (255, 60, 60, 165),
 'E2F3__L01': (255, 60, 60, 173),
 'KLF1__L01': (255, 60, 60, 163),
 'NR3C1__L01': (255, 60, 60, 168),
 'ADAM9__L01': (255, 60, 60, 180),
 'MAP3K1__L01': (255, 60, 60, 147),
 'E2F7__L01': (255, 60, 60, 169),
 'KLF3__L01': (255, 60, 60, 126),
 'SAMD8__L01': (255, 60, 60, 147),
 'TFAP2C__L01': (255, 60, 60, 142),
 'SEMA3C__L01': (255, 60, 60, 146),
 'KLF6__L01': (255, 60, 60, 150),
 'ZBTB7B__L01': (255, 60, 60, 148),
 'SOX11__L01': (255, 60, 60, 158),
 'E2F6__L01': (255, 60, 60, 123),
 'KLF10__L01': (255, 60, 60, 148),
 'KLF15__L01': (255, 60, 60, 146),
 'ZBTB14__L01': (255, 60, 60, 131),
 'TMEM178A__L01': (255, 60, 60, 126),
 'ETV1__L01': (255, 60, 60, 130),
 'ATF3__L01': (255, 60, 60, 143),
 'ETV4__L01': (255, 60, 60, 125),
 'EXOC6B__L01': (255, 60, 60, 124),
 'IL17D__L01': (255, 60, 60, 141),
 'OVOL2__L01': (255, 60, 60, 123),
 'TRIM46__L01': (255, 60, 60, 132),
 'ZNF668__L01': (255, 60, 60, 133),
 'PDE4D__L01': (255, 60, 60, 123),
 'KLF5__L01': (255, 

In [7]:
# link colors
edge_col_stacked = dict()
# check node colors to set edge color if both nodes have same color
for edge in G_stacked.edges(data=True):
    u, v = edge[0], edge[1]
    #filter for type "stacked"
    if edge[2]['type'] == "stacked":
        if col_stacked[u][:3] == col_stacked[v][:3]: # compare only r, g, b values
            edge_col_stacked[(u,v)] = col_stacked[u]
        else: # make grey 
            edge_col_stacked[(u,v)] = (100,100,100,100)

    else: # invisible for interlayer edges
        edge_col_stacked[(u,v, edge[2]['type'])] = (0,0,0,0)

nx.set_edge_attributes(G_stacked, edge_col_stacked, 'linkcolor')

In [8]:
# INTERLAYER 
G_stacked_inter = nx.Graph()
G_stacked_inter.add_edges_from([(u, v, {"type": t}) for u, v, t in l_edges_inter])
G_stacked_inter.add_nodes_from(df_nodes_stacked['Node ID'].tolist())

# positions
pos_stacked_inter = dict(zip(G_stacked_inter.nodes(),list(zip(df_nodes_stacked['x'], df_nodes_stacked['y'], df_nodes_stacked['z']))))
nx.set_node_attributes(G_stacked_inter, pos_stacked_inter, 'pos')

# colors
col_stacked_inter = dict(zip(G_stacked_inter.nodes(),list(zip(df_nodes_stacked['r'], df_nodes_stacked['g'], df_nodes_stacked['b'], df_nodes_stacked['a']))))
nx.set_node_attributes(G_stacked_inter, col_stacked_inter, 'nodecolor')

print("number of nodes in actin graph: ", len(G_stacked_inter.nodes()))
print("number of edges in actin graph: ", len(G_stacked_inter.edges()))

number of nodes in actin graph:  4744
number of edges in actin graph:  9345


In [9]:
# link colors
edge_col_inter = dict()
# check node colors to set edge color if both nodes have same color
for edge in G_stacked_inter.edges(data=True):
    u, v = edge[0], edge[1]
    #filter for type "stacked"
    if edge[2]['type'] == "interlayer":
        if col_stacked_inter[u][:3] == col_stacked_inter[v][:3]: # compare only r, g, b values
            edge_col_inter[(u,v)] = col_stacked_inter[u]
        else: # make grey 
            edge_col_inter[(u,v)] = (100,100,100,100)

    else: # invisible for interlayer edges
        edge_col_inter[(u,v, edge[2]['type'])] = (0,0,0,0)

nx.set_edge_attributes(G_stacked_inter, edge_col_inter, 'linkcolor')

In [10]:
G_stacked.graph['projectname'] = "TempPPI"
G_stacked.graph['info'] = "A temporarl PPI graph for testing purposes."

G_stacked.graph["layoutname"] = '01_stacked'
G_stacked_inter.graph["layoutname"] = '02_interlayer'

In [11]:
G_list = [G_stacked, G_stacked_inter]

## create VR basis project 

In [12]:
nx2j.create_project(G_list)

Successfully created the directory static/projects/TempPPI 
PROGRESS: loaded graph JSON...
PROGRESS: stored graph data...
PROGRESS: stored layouts...
PROGRESS: stored node info...
PROGRESS: made node position textures...
PROGRESS: made textures for node colors...
PROGRESS: made textures for links...
PROGRESS: made textures for linkcolors...
PROGRESS: made textures for linkcolors...
PROGRESS: writing json files for project and nodes...
Project created successfully.


## Realtime scenes

In [13]:
import networkx as nx 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

In [14]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()
client.connect()
#client.disconnect()

In [15]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'AE_Memes_2022'),
 (1, 'ARS23_memes'),
 (2, 'ByzNet-1400-people-only'),
 (3, 'ByzNet_PxL'),
 (4, 'CDK5'),
 (5, 'CircLadderGraph-xsmall'),
 (6, 'diffusion'),
 (7, 'JSON_autocore'),
 (8, 'Microplastics_HumanHealth'),
 (9, 'Pesticides_HumanHealth'),
 (10, 'PG_NEW'),
 (11, 'Powergrid_Europe'),
 (12, 'PPI_brain_infarction'),
 (13, 'PPI_joel_daniel_aryan_C'),
 (14, 'PPI_networkcartoGRAPHs'),
 (15, 'SpatialPPI'),
 (16, 'SpatialPPI_'),
 (17, 'SpatialPPI_onescene'),
 (18, 'Sphere_Torus'),
 (19, 'Teapot'),
 (20, 'TempPPI'),
 (21, 'TheMandelbulb_edges')]

In [16]:
# select a project to work with
sel_id = 20
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()
session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)

tools = AnalysisToolkit(session, tex_gen, syncer, file_mgr)

Session Graph loaded from project folder. 
Project name:  TempPPI
Data: Nodes: 4744 Links: 9345


In [17]:
# reconstruct Graph 
G = session.graph
print("number of nodes in loaded graph: ", len(G.nodes()))
print("number of edges in loaded graph: ", len(G.edges()))

number of nodes in loaded graph:  4744
number of edges in loaded graph:  9345
